In [47]:
import pandas as pd
from pathlib import Path
import sys
from neo4j import GraphDatabase, ResultSummary
from paths import DISC_STANDARDS_RAW
sys.path.insert(1, r"..\config")

In [ ]:
# URI examples: "neo4j://localhost", "neo4j+s://xxx.databases.neo4j.io"
URI = "neo4j://localhost"
AUTH = ("neo4j", "esgdisclosurekg")

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()

In [9]:
def run_cypher_query(query: str) -> ResultSummary:
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        summary = driver.execute_query(query).summary
    return summary.counters.nodes_created

In [ ]:
def bulk_import_csv_to_neo4j(driver, batch_size=1000):
    """
    Efficiently import a CSV file into Neo4j using Cypher UNWIND batching.
    
    Expected CSV columns:
      source, target, relationship

    Example:
      CompanyA, CO2 Emissions, REDUCED
      CompanyB, Water Usage, INCREASED
    """

    # Step 1: Read the CSV into memory
    df = filtered_df

    # required_cols = {"source", "target", "relationship"}
    # if not required_cols.issubset(df.columns):
    #     raise ValueError(f"CSV must contain columns: {required_cols}")

    cypher_query = """
      UNWIND $rows AS row
      MERGE (std:Standard {name: row.ESRS})
      MERGE (dr:DisclosureRequirement {name: row.DisclosureRequirement})
      MERGE (dp:Datapoint {id: row.ID, description: row.DatapointName, data_type: row.DatapointType, conditional_or_alternative_dp: row.ConditionalOrAlternativeDP, paragraph: row.Paragraph, ar: row.AR})
      SET dp.unit = row.Unit
      MERGE (std)-[:HAS_DISCLOSURE]->(dr)
      MERGE (dr)-[:HAS_DATAPOINT]->(dp)
      """

    # Step 3: Run in batches
    with driver.session() as session:
        for i in range(0, len(df), batch_size):
            batch = df.iloc[i:i+batch_size].to_dict("records")
            session.run(cypher_query, {"rows": batch})
            print(f"✅ Inserted rows {i}–{i + len(batch)}")

    driver.close()
    print(f"Imported {len(df)} relationships total into Neo4j.")


In [38]:
def get_filtered_df(raw: pd.DataFrame) -> pd.DataFrame:
    req_keys = [
    'ID',
    'ESRS',
    'DR',
    'Paragraph',
    'Related AR',
    'Name',
    'Data Type',
    'Conditional or alternative DP'
      ]
    
    filtered_df = raw.loc[:, req_keys]
    filtered_df =filtered_df.apply(lambda x: x.str.strip())
    filtered_df.rename(columns={"Related AR": "AR",\
                                 "Name": "DatapointName", "Data Type": "DatapointType",\
                                      "Conditional or alternative DP": "ConditionalOrAlternativeDP", \
                                        "DR": "DisclosureRequirement"}, inplace=True)
    filtered_df.fillna('NA', inplace=True)
    return filtered_df

In [ ]:
for sheet in ['ESRS E1', 'ESRS E2', 'ESRS E3', 'ESRS E4', 'ESRS E5', \
              'ESRS S1', 'ESRS S2', 'ESRS S3', 'ESRS S4','ESRS G1', 'ESRS 2', 'ESRS 2 MDR']:
    raw = pd.read_excel(DISC_STANDARDS_RAW / "EFRAG-IG-3-List-of-ESRS-Data-Points.xlsx", skiprows=1,sheet_name=sheet)
    filtered_df = get_filtered_df(raw)
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        bulk_import_csv_to_neo4j(driver)

✅ Inserted rows 0–217
Imported 217 relationships total into Neo4j.
✅ Inserted rows 0–72
Imported 72 relationships total into Neo4j.
✅ Inserted rows 0–51
Imported 51 relationships total into Neo4j.
✅ Inserted rows 0–125
Imported 125 relationships total into Neo4j.
✅ Inserted rows 0–67
Imported 67 relationships total into Neo4j.
✅ Inserted rows 0–198
Imported 198 relationships total into Neo4j.
✅ Inserted rows 0–71
Imported 71 relationships total into Neo4j.
✅ Inserted rows 0–69
Imported 69 relationships total into Neo4j.
✅ Inserted rows 0–69
Imported 69 relationships total into Neo4j.
✅ Inserted rows 0–55
Imported 55 relationships total into Neo4j.
✅ Inserted rows 0–146
Imported 146 relationships total into Neo4j.
✅ Inserted rows 0–44
Imported 44 relationships total into Neo4j.


### Converting the ESRS E1 set to Graph

- Make the nodes and relationships in arrows, and generate Cypher query for the same

### Converting to JSON

In [ ]:
nested = (
    filtered_df.groupby(["ESRS", "DR", "Paragraph"], dropna=False)
      .apply(lambda g: g[["Related AR", "Name", "Data Type", "Conditional or alternative DP"]]
             .rename(columns={
                 "Related AR": "AR",
                 "Data Type": "datatype",
                 "Conditional or alternative DP": "dp_type"
                 })
             .to_dict(orient="records"), include_groups=False)
      .reset_index(name="entries")
)

In [11]:
level1 = (
    nested.groupby("ESRS")
    .apply(lambda g: g.groupby("DR")
           .apply(lambda h: dict(zip(h["Paragraph"], h["entries"])), include_groups=False)
           .to_dict(), include_groups=False)
    .to_dict()
)

- If a property describes the nature, context, or conditions of the connection, it belongs on the relationship.

- If a property describes the inherent identity of the entity, it belongs on the node.